In [ ]:
def  Regressor_Predict(giorni_da_eliminare,giorni_test,giorni_train,freq):

ticker='BTCUSDT' 
freq=freq
        


init = 1
best_regressor = ''	

# Number of datapoints to lookahead
lookahead = 1	# predict the price for this datapoint

# Choose to see the best_regressor calculations in verbose mode
verbose = 0

# TSFresh rolling window size

max_window_size = 60	# max length of the rolled window for feature extract
min_window_size = 5	# minimum number of days for a rolling window


        
if init==1:

	init_regressors(ticker)


lookback = str(2000)+" days"
df=load_data(ticker,freq,lookback)
df_melted = pd.DataFrame() df_melted["timestamp"] = df.index df_melted["close"] = df['close'].values df_melted['Symbols'] = ticker df_melted.tail(3)



          df_rolled = roll_time_series(df_melted, column_id="Symbols",
         column_sort="timestamp", max_timeshift=max_window_size, 
         min_timeshift=min_window_size)






     X = extract_features(df_rolled.drop("Symbols", axis=1), column_id="id",          
     column_sort="timestamp", column_value="close", impute_function=impute,
     show_warnings=False)


      X = X.set_index(X.index.map(lambda x: x[1]), drop=True)
      X.index.name = "last_timestamp"


      y = df_melted.set_index("timestamp").sort_index().close.shift(-1)


      X['y'] = y



      def cross_validation(dataframe, giorni_da_eliminare, giorni_test, giorni_train):
                     if giorni_da_eliminare == 0:
                                 df = dataframe
                         else:
                                   df = dataframe.iloc[:-giorni_da_eliminare]
                                   df_test = df[-giorni_test:]
                                    start_train = -giorni_train-giorni_test
                                    df_train = df[start_train:-giorni_test]
        return df_test, df_train



        test_data, train_data = cross_validation(X, giorni_da_eliminare, giorni_test,                giorni_train)



         X_train = train_data.drop(columns=['y'])
         y_train = train_data['y']

         X_test = test_data.drop(columns=['y'])
         y_test = test_data['y']


         y = pd.concat([y_train, y_test])







test_start = X_test.index[0].date()

X_train_selected = select_features(X_train, y_train)



try:
        with open(f'regressor_min_mae_{ticker}.csv') as f:
            best_regressor=f.read()
        print ('Last saved regression method:',best_regressor)
except:
        pass


# For cleaner output we suppress warnings that occur during testing of the estimators
 from warnings import simplefilter
 from sklearn.exceptions import ConvergenceWarning
 simplefilter("ignore", category=FutureWarning)
 simplefilter("ignore", category=RuntimeWarning)
 simplefilter("ignore", category=ConvergenceWarning)
 simplefilter("ignore", category=UserWarning)


    # Check and proceed if init=1 or no best_regressor was found
              if (best_regressor =='') | init==1:

        # Create a list for regressors with too bad performance
              new_removed_regressors=[]
              removed_regressors=pd.DataFrame()



        # Calculate all the estimator metrics
        MAE,MSE,RMSE,R2,removed_regressors,
                    new_removed_regressors=get_best_regressor(X_train_selected,
                    X_test,y_train,
                    y,test_start,ticker,verbose)




        # Write the results to this file
        with open(f'regressor_min_mse_{ticker}.csv','w') as f:
            f.write(min(MSE, key=MSE.get))

        # Define the best estimator by chosing the one with the smallest MSE
        best_regressor=min(MSE, key=MSE.get)

              


        # Create a dataframe with the excluded estimators and write it to a file, so we can exclude them in future runs 
        if new_removed_regressors!='None':

                  new_removed_regressors=pd.DataFrame(new_removed_regressors,
                  columns=['name'])
                  removed_regressors=pd.concat([removed_regressors,
                  new_removed_regressors])
                  removed_regressors.to_csv(f'removed_regressors_{ticker}.csv',
                  'a',index=False)




# Load the estimator model and store its name
          estimators=all_estimators()

          for est_name, estimator in estimators:    
         	 	if str(best_regressor)==est_name:
         	 	best_regressor=estimator()
         		 name=est_name


# Print out the five best estimators
          df_best_regressors=pd.DataFrame.from_dict({'MSE': MSE, 'RMSE' : RMSE, 
          'R2_score': R2})
          df_best_regressors = df_best_regressors.sort_values(by="MSE",
          ascending=True).head().reset_index()
          df_styled=df_best_regressors.style.apply(lambda x: ['background-color: 
          lightgreen' if  x.name == 0 else '' 
          for i in x], axis=1)

          display(df_styled)

          best_model = str(best_regressor)


	

	    # Train the estimator on the training set
          best_regressor.fit(X_train_selected, y_train)

            # Prepare the test data
          X_test_selected = X_test[X_train_selected.columns]









        y_pred = pd.Series(best_regressor.predict(X_test_selected),
        index=X_test_selected.index)

        test_results=pd.concat([y_test,y_pred],axis=1)
        test_results.columns=['close','prediction']
        test_results['best_model'] = best_model

        test_results['giorni_eliminati'] = giorni_da_eliminare
         



         import os
         os.remove('removed_regressors_BTCUSDT.csv')
         os.remove('regressor_min_mse_BTCUSDT.csv')
         os.remove('regressors_BTCUSDT.csv')
    
         return test_results


















